# Feature Analysis — What the Model Actually Uses

This notebook explains every feature in plain English:
- What it measures clinically
- Normal ranges
- Distribution in our data
- How it differs between sepsis and non-sepsis patients
- Missing data rates

Run all cells top to bottom.

In [ ]:
import os
from pathlib import Path
os.chdir(Path.cwd().parent)
print(f'Working directory: {Path.cwd()}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

df = pd.read_parquet('data/processed/features.parquet')
print(f'Dataset: {len(df):,} ICU stays | {df["sepsis_label"].mean():.1%} sepsis')
print(f'Features: {df.shape[1]} columns')
df.head(3)

## 1. Missing Data Overview

Not every patient has every test. Understanding missingness tells us which features are routinely available vs ordered only when there's a concern.

In [ ]:
feature_cols = [c for c in df.columns if c not in ['stay_id','hadm_id','sepsis_label']]
missing = df[feature_cols].isnull().mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(12, 8))
missing.plot(kind='barh', ax=ax, color=['#e74c3c' if v > 50 else '#f39c12' if v > 20 else '#27ae60' for v in missing])
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Data by Feature\n(green=<20%, orange=20-50%, red=>50%)')
ax.axvline(20, color='orange', linestyle='--', alpha=0.5)
ax.axvline(50, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print('\nFeatures with >50% missing (ordered only when clinically indicated):')
print(missing[missing > 50].to_string())
print('\nFeatures with <20% missing (routinely measured):')
print(missing[missing < 20].to_string())

## 2. Sepsis vs Non-Sepsis: Key Differences

For each feature, we compare the distribution between sepsis and non-sepsis patients.
A large gap = the feature is a strong signal. A small gap = less useful.

In [ ]:
sepsis = df[df['sepsis_label'] == 1]
non_sepsis = df[df['sepsis_label'] == 0]

# Key clinical features to compare
key_features = [
    ('lactate_last',      'Lactate (mmol/L)',        0,  15,  'Sepsis marker. >2.0 = concern, >4.0 = shock'),
    ('wbc_last',          'WBC (K/µL)',               0,  30,  'White blood cells. High OR low suggests infection'),
    ('creatinine_last',   'Creatinine (mg/dL)',       0,  10,  'Kidney function. Rising = organ dysfunction'),
    ('map_min',           'MAP min (mmHg)',           30, 110, 'Blood pressure. <65 triggers vasopressor consideration'),
    ('heart_rate_mean',   'Heart Rate mean (bpm)',    40, 150, 'Tachycardia (>100) is a sepsis criterion'),
    ('resp_rate_mean',    'Resp Rate mean (/min)',    5,  45,  'RR>22 is a qSOFA criterion for sepsis'),
    ('bilirubin_last',    'Bilirubin (mg/dL)',        0,  10,  'Liver function (SOFA score component)'),
    ('platelets_last',    'Platelets (K/µL)',         0,  400, 'Coagulation. Low = DIC risk in sepsis'),
    ('bicarbonate_last',  'Bicarbonate (mEq/L)',      10, 35,  'Acid-base. Low = metabolic acidosis'),
    ('spo2_min',          'SpO2 min (%)',             80, 100, 'Oxygen saturation. Low = respiratory compromise'),
]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, (feat, label, lo, hi, note) in enumerate(key_features):
    ax = axes[i]
    s_vals = sepsis[feat].dropna().clip(lo, hi)
    n_vals = non_sepsis[feat].dropna().clip(lo, hi)
    
    ax.hist(n_vals, bins=40, alpha=0.5, color='#3498db', density=True, label='No Sepsis')
    ax.hist(s_vals, bins=40, alpha=0.5, color='#e74c3c', density=True, label='Sepsis')
    
    # Means
    ax.axvline(n_vals.mean(), color='#2980b9', linestyle='--', linewidth=1.5)
    ax.axvline(s_vals.mean(), color='#c0392b', linestyle='--', linewidth=1.5)
    
    # T-test p-value
    _, p = stats.ttest_ind(s_vals, n_vals, equal_var=False)
    ax.set_title(f'{label}\np={p:.2e}', fontsize=9)
    ax.set_xlim(lo, hi)
    ax.legend(fontsize=7)
    
    # Add note
    ax.text(0.02, 0.98, note, transform=ax.transAxes,
            fontsize=6, verticalalignment='top', style='italic', color='gray',
            wrap=True)

plt.suptitle('Feature Distributions: Sepsis vs Non-Sepsis (red=sepsis, blue=non-sepsis)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Effect Size — Which Features Discriminate Best?

Cohen's d measures how different the sepsis and non-sepsis groups are for each feature.
d > 0.8 = large effect (very useful feature), d < 0.2 = small effect (less useful)

In [ ]:
def cohens_d(group1, group2):
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(), group2.var()
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    if pooled_std == 0: return 0
    return abs((group1.mean() - group2.mean()) / pooled_std)

effect_sizes = {}
for col in feature_cols:
    s = sepsis[col].dropna()
    n = non_sepsis[col].dropna()
    if len(s) > 30 and len(n) > 30:
        effect_sizes[col] = cohens_d(s, n)

eff_df = pd.Series(effect_sizes).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#e74c3c' if v > 0.8 else '#f39c12' if v > 0.5 else '#3498db' for v in eff_df]
eff_df.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0.2, color='blue', linestyle=':', alpha=0.7, label='Small effect (0.2)')
ax.axvline(0.5, color='orange', linestyle=':', alpha=0.7, label='Medium effect (0.5)')
ax.axvline(0.8, color='red', linestyle=':', alpha=0.7, label='Large effect (0.8)')
ax.set_xlabel("Cohen's d (effect size)")
ax.set_title("Top 20 Features by Discriminative Power\n(red=large effect, orange=medium, blue=small)")
ax.legend()
plt.tight_layout()
plt.show()

print('\nTop 10 most discriminative features:')
for feat, d in eff_df.head(10).items():
    print(f'  {feat:35s} d = {d:.3f}')

## 4. What Is Actually Measured Daily in ICU?

Some tests are done every shift. Others only when the doctor orders them.
This affects how many patients have each value available.

In [ ]:
availability = (df[feature_cols].notnull().mean() * 100).sort_values(ascending=False)

print('=== ROUTINELY MEASURED (>80% of patients have this) ===')
print('These are the features the model can rely on most:')
for feat, pct in availability[availability > 80].items():
    print(f'  {feat:40s} {pct:.1f}%')

print('\n=== SOMETIMES MEASURED (20-80%) ===')
print('Ordered when clinically indicated:')
for feat, pct in availability[(availability >= 20) & (availability <= 80)].items():
    print(f'  {feat:40s} {pct:.1f}%')

print('\n=== RARELY MEASURED (<20%) ===')
print('Only in specific clinical situations:')
for feat, pct in availability[availability < 20].items():
    print(f'  {feat:40s} {pct:.1f}%')

## 5. Sepsis-3 Clinical Criteria — How Well Does Our Data Cover Them?

Sepsis-3 is defined as life-threatening organ dysfunction from infection.
Diagnosed using SOFA score ≥2. qSOFA is the quick bedside screen.

**qSOFA (quick bedside screen — 1 point each):**
- RR ≥ 22 breaths/min
- Altered mentation (GCS < 15)
- SBP ≤ 100 mmHg

**SOFA score components:**
- Respiratory: PaO2/FiO2 ratio
- Coagulation: Platelets
- Liver: Bilirubin
- Cardiovascular: MAP + vasopressors
- Neurological: GCS
- Renal: Creatinine + urine output

In [ ]:
# Compute a simplified SOFA score from available features
def simplified_sofa(row):
    score = 0
    # Coagulation (platelets)
    p = row.get('platelets_last', np.nan)
    if not np.isnan(p):
        if p < 20: score += 4
        elif p < 50: score += 3
        elif p < 100: score += 2
        elif p < 150: score += 1
    # Liver (bilirubin)
    b = row.get('bilirubin_last', np.nan)
    if not np.isnan(b):
        if b >= 12: score += 4
        elif b >= 6: score += 3
        elif b >= 2: score += 2
        elif b >= 1.2: score += 1
    # Cardiovascular (MAP)
    m = row.get('map_min', np.nan)
    if not np.isnan(m):
        if m < 65: score += 1
    # Renal (creatinine)
    c = row.get('creatinine_last', np.nan)
    if not np.isnan(c):
        if c >= 5: score += 4
        elif c >= 3.5: score += 3
        elif c >= 2: score += 2
        elif c >= 1.2: score += 1
    return score

df['sofa_proxy'] = df.apply(simplified_sofa, axis=1)

print('SOFA proxy score distribution in sepsis vs non-sepsis patients:')
print(f'  Sepsis patients     — mean SOFA: {df[df["sepsis_label"]==1]["sofa_proxy"].mean():.2f}')
print(f'  Non-sepsis patients — mean SOFA: {df[df["sepsis_label"]==0]["sofa_proxy"].mean():.2f}')

fig, ax = plt.subplots(figsize=(8, 4))
for label, color, name in [(1, '#e74c3c', 'Sepsis'), (0, '#3498db', 'No Sepsis')]:
    subset = df[df['sepsis_label'] == label]['sofa_proxy']
    ax.hist(subset, bins=range(0, 12), alpha=0.6, color=color, density=True, label=name)
ax.axvline(2, color='black', linestyle='--', label='SOFA ≥2 = Sepsis threshold')
ax.set_xlabel('Simplified SOFA Score')
ax.set_title('SOFA Score Distribution: Sepsis vs Non-Sepsis')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Correlation Between Features

Highly correlated features carry redundant information. The model handles this fine, but good to understand.

In [ ]:
# Use only features with <50% missing
good_features = [c for c in feature_cols if df[c].isnull().mean() < 0.5]
corr = df[good_features].corr()

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(good_features)))
ax.set_yticks(range(len(good_features)))
ax.set_xticklabels(good_features, rotation=90, fontsize=7)
ax.set_yticklabels(good_features, fontsize=7)
ax.set_title('Feature Correlation Matrix (features with <50% missing)', fontsize=12)
plt.tight_layout()
plt.show()

print('\nHighly correlated pairs (|r| > 0.8):')
for i in range(len(good_features)):
    for j in range(i+1, len(good_features)):
        r = corr.iloc[i,j]
        if abs(r) > 0.8:
            print(f'  {good_features[i]:35s} <-> {good_features[j]:35s} r={r:.2f}')

## 7. Model Feature Importances

Which features does the trained model consider most important overall?

In [ ]:
import joblib
artifact = joblib.load('models/lightgbm_sepsis.pkl')
model = artifact['model']
feat_cols = artifact['feature_cols']

importances = model.feature_importances_
imp_df = pd.Series(importances, index=feat_cols).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 7))
imp_df.plot(kind='barh', ax=ax, color='#2980b9')
ax.set_xlabel('Feature Importance (mean decrease in impurity)')
ax.set_title('Top 20 Model Feature Importances')
plt.tight_layout()
plt.show()

print('\nTop 10 features by model importance:')
for feat, imp in imp_df.head(10).items():
    clinical_note = {
        'lactate_last': 'KEY sepsis marker — metabolic stress indicator',
        'lactate_mean': 'KEY sepsis marker — sustained elevation is worse',
        'creatinine_last': 'Kidney function — SOFA renal component',
        'map_min': 'Blood pressure — <65 mmHg triggers intervention',
        'wbc_last': 'Infection marker — high AND low both signal sepsis',
        'heart_rate_mean': 'Tachycardia — one of qSOFA criteria',
        'resp_rate_mean': 'Tachypnea — RR>22 is qSOFA criterion',
        'bicarbonate_last': 'Acid-base — low = metabolic acidosis from poor perfusion',
        'platelets_last': 'Coagulation — SOFA component, low = DIC risk',
    }.get(feat, 'See clinical reference')
    print(f'  {feat:35s} {imp:.4f}  — {clinical_note}')

## 8. A Concrete Example — What a HIGH Risk Alert Looks Like

Let's look at an actual high-risk patient from our dataset and understand why the model flagged them.

In [ ]:
from src.model.predict import predict_batch

results = predict_batch(df, artifact)
high_risk_sepsis = results[(results['risk_score'] > 0.7) & (results['sepsis_label'] == 1)]

if len(high_risk_sepsis) > 0:
    example = high_risk_sepsis.iloc[0]
    stay_id = example['stay_id']
    
    print(f'Example HIGH-RISK patient (Stay ID: {stay_id})')
    print(f'Risk Score: {example["risk_score"]:.3f} | True Label: SEPSIS')
    print()
    
    # Show their key values with clinical context
    feat_row = df[df['stay_id'] == stay_id].iloc[0]
    
    checks = [
        ('lactate_last',     'Lactate',       '<2.0 mmol/L'),
        ('wbc_last',         'WBC',           '4-11 K/µL'),
        ('creatinine_last',  'Creatinine',    '<1.2 mg/dL'),
        ('map_min',          'MAP (min)',      '≥65 mmHg'),
        ('heart_rate_mean',  'Heart Rate',    '60-100 bpm'),
        ('resp_rate_mean',   'Resp Rate',     '12-20/min'),
        ('bilirubin_last',   'Bilirubin',     '<1.2 mg/dL'),
        ('bicarbonate_last', 'Bicarbonate',   '22-29 mEq/L'),
        ('platelets_last',   'Platelets',     '150-400 K/µL'),
    ]
    
    print(f'{"Measurement":25s} {"Value":12s} {"Normal Range":20s} {"Status"}')
    print('-' * 75)
    for feat, name, normal in checks:
        val = feat_row.get(feat, np.nan)
        if not (isinstance(val, float) and np.isnan(val)):
            print(f'{name:25s} {val:12.2f} {normal:20s}')
        else:
            print(f'{name:25s} {"not measured":12s} {normal:20s}')